# `cluster_topics` — User Guide

Daily topic clustering pipeline that discovers emergent market narratives from RSS feed articles and tracks their frequency as a trading signal.

```
cluster_topics.py
│
├── Core clustering
│   ├── extract_window_vectors(date, window_days)  → (np.ndarray, pd.DataFrame)
│   ├── reduce_dimensions(vectors, n_components)  → np.ndarray
│   └── run_hdbscan(X, ...)                        → (labels, noise_ratio)
│
├── Topic continuity
│   ├── compute_centroids(vectors, labels)         → dict[int, ndarray]
│   ├── match_topics(new_centroids, prior, ...)    → dict[int, dict]
│   ├── load_centroids(path) / save_centroids(...)→ dict / None
│   └── load_label_cache(path) / save_label_cache → dict / None
│
├── LLM labeling (requires OPENAI_API_KEY)
│   └── get_label(topic_id, cache, articles, ...)  → str
│
├── Time-series output
│   └── append_trends(run_date, rows, path)        → None | DuplicateDateError
│
├── Signal generation
│   ├── compute_spike(topic_id, trends, date, ...) → float | None
│   └── get_emerging_topics(date, trends, ...)     → list[dict]
│
└── Full pipeline (requires OPENAI_API_KEY + FAISS store)
    └── run(target_date, ...)                      → summary dict
```

**Prerequisites:**
- `OPENAI_API_KEY` in environment — required to load the FAISS store.
- `data/vectorstore/feeds/` — FAISS index built by `embed_feeds.py`.
- Sections 5–10 can run once section 2 has loaded `VECTORS` and `META`.

## Sections
1. [Setup](#1-setup)
2. [Load data from FAISS](#2-load-data-from-faiss)
3. [Core clustering](#3-core-clustering)
4. [Topic continuity](#4-topic-continuity)
5. [Centroid and label cache I/O](#5-centroid-and-label-cache-io)
6. [Time-series output](#6-time-series-output)
7. [Signal generation](#7-signal-generation)
8. [LLM labeling (API-key gated)](#8-llm-labeling)
9. [Full pipeline](#9-full-pipeline)
10. [Error handling reference](#10-error-handling-reference)


---
## 1 — Setup

Import project modules and configure logging.

In [ ]:
import json
import logging
import os
import sys
import tempfile
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..") if Path("../cluster_topics.py").exists() else Path(".")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cluster_topics import (
    ClusteringAborted, DuplicateDateError,
    append_trends, compute_centroids, compute_spike,
    extract_window_vectors, get_emerging_topics, get_label,
    load_centroids, load_label_cache, match_topics,
    reduce_dimensions, run_hdbscan, save_centroids, save_label_cache,
)
from constants import (
    CLUSTER_MAX_NOISE_RATIO, CLUSTER_MIN_CLUSTERS,
    CLUSTER_MIN_SAMPLES, CLUSTER_MIN_SIZE, CLUSTER_WINDOW_DAYS,
    TOPIC_CENTROIDS_FILE, TOPIC_TRENDS_FILE,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
    force=True,
)

RUN_DATE = date(2026, 3, 23)  # target date used throughout

print(f"CLUSTER_WINDOW_DAYS     : {CLUSTER_WINDOW_DAYS}")
print(f"CLUSTER_MIN_SIZE        : {CLUSTER_MIN_SIZE}")
print(f"CLUSTER_MIN_SAMPLES     : {CLUSTER_MIN_SAMPLES}")
print(f"CLUSTER_MAX_NOISE_RATIO : {CLUSTER_MAX_NOISE_RATIO}")
print(f"CLUSTER_MIN_CLUSTERS    : {CLUSTER_MIN_CLUSTERS}")


---
## 2 — Load data from FAISS

`extract_window_vectors` loads the 45-day rolling window of article embeddings directly from the existing FAISS index — **no re-embedding**. It returns:

- `VECTORS` — `np.ndarray` shape `(n, 1536)`, float32
- `META` — `pd.DataFrame` with columns `id, date, title, link, guid`; row `i` corresponds to `VECTORS[i]`

**Requires** `OPENAI_API_KEY` (to construct the `OpenAIEmbeddings` instance needed by `FAISS.load_local`) and `data/vectorstore/feeds/index.faiss`.


In [ ]:
VECTORS, META = extract_window_vectors(RUN_DATE, window_days=CLUSTER_WINDOW_DAYS)
META["date"] = pd.to_datetime(META["date"])

print(f"VECTORS shape  : {VECTORS.shape}")
print(f"META shape     : {META.shape}")
print(f"Date range     : {META['date'].min().date()} -> {META['date'].max().date()}")
print(f"Window articles: {len(META)}")
print(f"\nSample titles:")
for t in META["title"].head(5).tolist():
    print(f"  {str(t)[:90]}")


---
## 3 — Core clustering

### `reduce_dimensions(vectors, n_components=50)`

Wraps `sklearn.decomposition.PCA` with `random_state=42` for determinism. Reduces 1 536 → 50 dims to remove noise axes before clustering.

### `run_hdbscan(X, min_cluster_size, min_samples, max_noise_ratio, min_clusters)`

| Parameter | Default | Effect |
|---|---|---|
| `min_cluster_size` | 10 | Minimum articles to form a cluster |
| `min_samples` | 3 | Controls cluster boundary conservatism |
| `max_noise_ratio` | 0.90 | Abort threshold — 60–70% noise is normal for news data |
| `min_clusters` | 3 | Abort if fewer clusters form |

Label `-1` = noise (article does not belong to any recurring narrative).


In [ ]:
X = reduce_dimensions(VECTORS, n_components=50)
print(f"Input shape  : {VECTORS.shape}")
print(f"Reduced shape: {X.shape}")

labels, noise_ratio = run_hdbscan(X, min_cluster_size=CLUSTER_MIN_SIZE,
                                   min_samples=CLUSTER_MIN_SAMPLES)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
sizes = pd.Series(labels[labels != -1]).value_counts().sort_values(ascending=False)

print(f"\nClusters found : {n_clusters}")
print(f"Noise ratio    : {noise_ratio:.1%}  ({(labels == -1).sum()} / {len(labels)} articles)")
print(f"\nCluster sizes (top 10):")
for cid, sz in sizes.head(10).items():
    print(f"  cluster {int(cid):>2} : {sz} articles")


In [ ]:
# Sample article titles per cluster -- shows real CNBC headlines
print("Sample article titles per cluster (top 5):")
for cid in sorted(set(labels)):
    if cid == -1:
        continue
    mask = labels == cid
    sample_titles = META[mask]["title"].head(5).tolist()
    count = int(mask.sum())
    print(f"\n  cluster {int(cid):>2} ({count} articles):")
    for t in sample_titles:
        print(f"    - {str(t)[:90]}")


---
## 4 — Topic continuity

HDBSCAN cluster IDs (0, 1, 2…) are **arbitrary and change every run**. Stable `topic_id` (UUID) is maintained by matching today’s centroids to prior centroids via cosine similarity (greedy, threshold=0.85).

### `compute_centroids(vectors, labels)`

Mean embedding vector per cluster. Noise (`-1`) is never a key.

### `match_topics(new_centroids, prior_topics, threshold=0.85, run_date)`

Greedy cosine-similarity assignment: builds an (n_new × n_prior) matrix, assigns pairs highest-first, each prior topic matches at most one new cluster. Unmatched clusters receive a fresh UUID.


In [ ]:
centroids = compute_centroids(VECTORS, labels)
print(f"Clusters with centroids: {len(centroids)}")
for cid, vec in sorted(centroids.items()):
    print(f"  cluster {int(cid):>2} : shape={vec.shape}, norm={np.linalg.norm(vec):.4f}")


In [ ]:
# Load prior centroids from the persistent store (data/topic_centroids.json)
# On first ever run this returns {}; after daily CI runs it contains real topic history.
prior_topics = load_centroids(TOPIC_CENTROIDS_FILE)
print(f"Prior topics loaded: {len(prior_topics)}")

topic_map = match_topics(centroids, prior_topics, threshold=0.85, run_date=RUN_DATE)

matched = sum(
    1 for e in topic_map.values()
    if e["topic_id"] in prior_topics
)
fresh = len(topic_map) - matched
print(f"Matched to prior topics : {matched}")
print(f"New topic IDs assigned  : {fresh}")
print()
for cid, entry in sorted(topic_map.items()):
    is_match = entry["topic_id"] in prior_topics
    status = "MATCHED" if is_match else "NEW    "
    label = entry.get("label", "") or "(unlabeled)"
    print(f"  cluster {int(cid):>2} [{status}] {entry['topic_id'][:8]}...  {label[:40]}")


---
## 5 — Centroid and label cache I/O

| Function | Reads / Writes | Safe on first run? |
|---|---|---|
| `load_centroids(path)` | JSON dict `{topic_id -> entry}` | Yes — returns `{}` if absent |
| `save_centroids(data, path)` | Atomic write via `.json.tmp` + `os.replace` | Yes |
| `load_label_cache(path)` | JSON dict `{topic_id -> label}` | Yes — returns `{}` if absent |
| `save_label_cache(cache, path)` | Atomic write via `.json.tmp` + `os.replace` | Yes |

Default paths are `data/topic_centroids.json` and `data/topic_labels.json` (configured in `constants.py`). Pass an explicit `path` to redirect for testing.


In [ ]:
import tempfile, pathlib

with tempfile.TemporaryDirectory() as tmpdir:
    c_path = pathlib.Path(tmpdir) / "topic_centroids.json"
    l_path = pathlib.Path(tmpdir) / "topic_labels.json"

    # First-run behaviour: files absent -> empty dicts
    assert load_centroids(c_path) == {}
    assert load_label_cache(l_path) == {}
    print("First run (no files): both return {} as expected")

    # Persist today's topic_map
    centroid_store = {entry["topic_id"]: entry for entry in topic_map.values()}
    save_centroids(centroid_store, c_path)
    print(f"Saved {len(centroid_store)} centroid entries")

    label_cache = {entry["topic_id"]: entry.get("label", "") for entry in topic_map.values()}
    save_label_cache(label_cache, l_path)
    print(f"Saved {len(label_cache)} label entries")

    # Round-trip verify
    assert set(load_centroids(c_path).keys()) == set(centroid_store.keys())
    assert set(load_label_cache(l_path).keys()) == set(label_cache.keys())
    print("Round-trip verified")

    # Atomic write: no .tmp left behind
    assert list(pathlib.Path(tmpdir).glob("*.tmp")) == []
    print("Atomic write confirmed: no .tmp files")


---
## 6 — Time-series output (`append_trends`)

`append_trends` writes the daily count per topic to an append-only TSV:

```
data/topic_trends.tsv
  date         topic_id    topic_label              article_count
  2026-03-22   3b4e...     Fed rate pause bets      34
  2026-03-23   3b4e...     Fed rate pause bets      29
```

- **Append-only**: existing rows are never modified
- **Atomic**: temp file + `os.replace` prevents partial writes
- **Idempotent guard**: `DuplicateDateError` if the same date is submitted twice


In [ ]:
import tempfile, pathlib

# Build trend rows from today's real clustering result
trend_rows = [
    {
        "date": str(RUN_DATE),
        "topic_id": entry["topic_id"],
        "topic_label": entry.get("label", ""),
        "article_count": int((labels == cid).sum()),
    }
    for cid, entry in topic_map.items()
]

with tempfile.TemporaryDirectory() as tmpdir:
    tsv_path = pathlib.Path(tmpdir) / "topic_trends.tsv"

    append_trends(RUN_DATE, trend_rows, tsv_path)
    df = pd.read_csv(tsv_path, sep="\t")
    print(f"Appended {len(df)} rows:")
    print(df.to_string(index=False))

    # Second append on a different date succeeds
    next_date = date(2026, 3, 24)
    append_trends(next_date, [{**r, "date": str(next_date)} for r in trend_rows], tsv_path)
    df2 = pd.read_csv(tsv_path, sep="\t")
    print(f"\nTwo-date file: {len(df2)} rows, {df2['date'].nunique()} unique dates")

    # Duplicate date raises DuplicateDateError
    try:
        append_trends(RUN_DATE, trend_rows, tsv_path)
    except DuplicateDateError as e:
        print(f"\nDuplicateDateError: {e}")


---
## 7 — Signal generation

### `compute_spike(topic_id, trends, target_date, lookback_days=7, min_history=3)`

Returns `today_count / mean(prior_N_days)` or `None` when there is insufficient history (< `min_history` prior days or zero rolling average).

### `get_emerging_topics(target_date, trends, min_article_count=5, spike_lookback=7)`

Filters to topics with `article_count >= min_article_count`, computes spike ratios, returns sorted list `[{topic_id, label, spike_ratio, article_count}, ...]`.

The cell below loads `data/topic_trends.tsv` — the real append-only file. Spike computation requires at least 3 days of history per topic; `None` is returned until enough data accumulates.


In [ ]:
trends_path = Path(str(PROJECT_ROOT)) / "data" / "topic_trends.tsv"

if trends_path.exists():
    trends_df = pd.read_csv(trends_path, sep="\t")
    print(f"Loaded topic_trends.tsv: {len(trends_df)} rows, {trends_df['date'].nunique()} dates")
    print(trends_df.head(10).to_string(index=False))

    signals = get_emerging_topics(RUN_DATE, trends_df, min_article_count=5)
    if signals:
        print(f"\nEmerging topics on {RUN_DATE}: {len(signals)}")
        for s in signals:
            print(f"  spike={s['spike_ratio']:.2f}  count={s['article_count']:>4}"                  f"  {s['label'][:50]}")
    else:
        print(f"\nNo spike signals for {RUN_DATE}")
        print("(spike needs >= 3 prior days per topic; run daily CI to build history)")
else:
    print(f"topic_trends.tsv not found at {trends_path}")
    print("Run: python cluster_topics.py   to generate it")


---
## 8 — LLM labeling (`get_label`)

`get_label` assigns a 3–5 word label to a topic via `gpt-4o-mini`. **Cache-first**: if the `topic_id` is already in `topic_labels.json`, the LLM is never called. In steady state: 0–3 API calls per day.

**Requires `OPENAI_API_KEY`.**


In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set -- skipping.")
else:
    # Pick the first unlabeled topic from today's clustering
    unlabeled = [
        (cid, entry) for cid, entry in topic_map.items()
        if not entry.get("label")
    ]
    if not unlabeled:
        print("All topics already have labels (from prior run).")
    else:
        cid, entry = unlabeled[0]
        tid = entry["topic_id"]
        member_titles = META[labels == cid]["title"].head(15).tolist()
        cache = {}
        label = get_label(tid, cache, member_titles)
        print(f"topic_id    : {tid}")
        print(f"label       : {label!r}")
        print(f"Sample headlines used:")
        for t in member_titles[:5]:
            print(f"  - {str(t)[:80]}")

    # Cache-hit demo: calling again never re-invokes the LLM
    if unlabeled:
        existing_tid = unlabeled[0][1]["topic_id"]
        def explode(_): raise RuntimeError("LLM called on cache hit!")
        cached_label = get_label(existing_tid, cache, [], llm_fn=explode)
        print(f"\nCache hit -> {cached_label!r}  (no LLM call)")


---
## 9 — Full pipeline (`run`)

`run()` executes all 7 steps for a single day. Uses temp directories so it does not overwrite `data/topic_trends.tsv` or other persistent files.

```python
summary = run(
    target_date    = date.today(),
    window_days    = 45,
    centroids_path = Path("data/topic_centroids.json"),
    labels_path    = Path("data/topic_labels.json"),
    trends_path    = Path("data/topic_trends.tsv"),
    clusters_dir   = Path("data/topic_clusters"),
    skip_labeling  = False,
)
```

**Requires** `OPENAI_API_KEY` + FAISS store.


In [ ]:
import pprint, tempfile
from cluster_topics import run, ClusteringAborted

if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set -- skipping.")
else:
    with tempfile.TemporaryDirectory() as tmpdir:
        td = pathlib.Path(tmpdir)
        try:
            summary = run(
                target_date    = RUN_DATE,
                centroids_path = td / "topic_centroids.json",
                labels_path    = td / "topic_labels.json",
                trends_path    = td / "topic_trends.tsv",
                clusters_dir   = td / "topic_clusters",
                skip_labeling  = False,
            )
            print("run() completed:")
            pprint.pprint(summary)
        except ClusteringAborted as e:
            print(f"ClusteringAborted: {e}")


---
## 10 — Error handling reference

| Function | Raises | When |
|---|---|---|
| `run_hdbscan` | `ClusteringAborted` | noise ratio > `max_noise_ratio` OR n_clusters < `min_clusters` |
| `append_trends` | `DuplicateDateError` | `run_date` already present in the TSV |
| `extract_window_vectors` | `AssertionError` | row count mismatch vectors vs metadata |
| `match_topics` | *(none)* | always returns (worst case: all new UUIDs) |
| `get_label` | *(none)* | LLM failure returns `"Unknown topic"` instead of raising |
| `load_centroids` | *(none)* | returns `{}` when file absent |
| `load_label_cache` | *(none)* | returns `{}` when file absent |

In practice:
- **`ClusteringAborted`**: degenerate run. CI treats exit code 2 as a warning.
- **`DuplicateDateError`**: delete the duplicate rows from `topic_trends.tsv` manually before re-running.


In [ ]:
import tempfile, pathlib, numpy as np

# --- ClusteringAborted: excessive noise ---
rng3 = np.random.default_rng(seed=42)
noise_data = rng3.standard_normal((300, 50)).astype(np.float32) * 5.0
try:
    run_hdbscan(noise_data, min_cluster_size=50, min_samples=10,
                max_noise_ratio=0.50, min_clusters=3)
except ClusteringAborted as e:
    print(f"ClusteringAborted (high noise): {e}")

# --- ClusteringAborted: too few clusters ---
centers2 = rng3.standard_normal((3, 50)).astype(np.float32) * 10
blob_data = np.vstack([
    centers2[i] + rng3.standard_normal((50, 50)).astype(np.float32) * 0.05
    for i in range(3)
])
try:
    run_hdbscan(blob_data, min_cluster_size=5, min_samples=2,
                max_noise_ratio=0.95, min_clusters=10)
except ClusteringAborted as e:
    print(f"ClusteringAborted (too few clusters): {e}")

# --- DuplicateDateError ---
with tempfile.TemporaryDirectory() as tmpdir:
    tsv_path = pathlib.Path(tmpdir) / "topic_trends.tsv"
    rows = [{"date": "2026-03-23", "topic_id": "t1",
             "topic_label": "Test", "article_count": 10}]
    append_trends(date(2026, 3, 23), rows, tsv_path)
    try:
        append_trends(date(2026, 3, 23), rows, tsv_path)
    except DuplicateDateError as e:
        print(f"DuplicateDateError: {e}")
